In [ ]:
from pathlib import Path
import os

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
os.chdir(REPO)      # all paths below are relative to the repository root

import json
from collections import Counter

import pandas as pd

from ct_iqa.data.ldct_iqac import VALID_EXTENSIONS
from ct_iqa.data.splits import load_manifest
from ct_iqa.experiment.metadata import environment_info
from ct_iqa.models.weights import sha256_file


def show(title, obj):
    """Print a section title, then the table as plain text (no interactive viewer)."""
    print(f"\n=== {title} ===")
    print(obj.to_string())

CHECKS = {}      # check name -> bool
DETAILS = {}     # check name -> explanation shown for every failed check


def check(name, ok, detail=""):
    """Record one check. `ok` must be a real bool; anything else counts as a failure."""
    CHECKS[name] = ok is True
    if ok is not True or detail:
        DETAILS[name] = detail if ok is True else (detail or "condition is False")


def guarded(name, fn):
    """Run `fn` (returns (ok, detail)); an exception is a recorded failure, never a skipped check."""
    try:
        ok, detail = fn()
        check(name, ok, detail)
    except Exception as e:
        check(name, False, f"{type(e).__name__}: {e}")

TRAIN_IMAGES = Path("data/raw/ldct_iqac/train/image")
TRAIN_LABELS = Path("data/labels/ldct_iqac/train.json")
MANIFEST = "experiments/study_v2/protocol/split_manifest.json"

In [ ]:
info = environment_info("auto")
show("Environment", pd.DataFrame({"value": info}))

In [ ]:
labels = {}
manifest = None


def labels_check():
    global labels
    labels = json.loads(TRAIN_LABELS.read_text(encoding="utf-8"))
    vals = list(labels.values())
    check("train labels = 1000", len(labels) == 1000, f"found {len(labels)}")
    check("labels are finite numbers", all(isinstance(v, (int, float)) and not isinstance(v, bool) and v == v for v in vals))
    check("train scores within [0, 4]", min(vals) >= 0 and max(vals) <= 4, f"range {min(vals)}..{max(vals)}")
    off = [k for k, v in labels.items() if abs(v * 5 - round(v * 5)) > 1e-6]
    check("all scores are multiples of 0.2", not off, f"{len(off)} off-grid, e.g. {off[:3]}")
    return True, ""


def manifest_check():
    global manifest
    manifest = load_manifest(MANIFEST, labels)        # raises if hash, overlap or coverage is wrong
    sizes = [len(manifest[k]) for k in ("train", "validation", "calibration")]
    check("split partitions 800/100/100", sizes == [800, 100, 100], f"sizes {sizes}")
    print("manifest sha256:", manifest["manifest_sha256"])
    return True, ""


guarded("train label file readable and valid", labels_check)
guarded("frozen split manifest verifies (hash, disjoint, complete)", manifest_check)
if manifest:
    show("Label statistics per partition (raw 0-4 scale)", pd.DataFrame(
        {p: pd.Series([labels[n] for n in manifest[p]]).describe() for p in ("train", "validation", "calibration")}).T.round(3))

In [ ]:
def filenames_check():
    entries = sorted(p for p in TRAIN_IMAGES.iterdir())
    files = [p.name for p in entries if p.is_file()]
    images = [n for n in files if Path(n).suffix.lower() in VALID_EXTENSIONS]
    unexpected = [n for n in files if Path(n).suffix.lower() not in VALID_EXTENSIONS]
    subdirs = [p.name for p in entries if p.is_dir()]
    stems = Counter(Path(n).stem.lower() for n in images)
    clashes = sorted(s for s, c in stems.items() if c > 1)
    no_label = sorted(set(images) - set(labels))
    no_image = sorted(set(labels) - set(images))
    in_manifest = set().union(*(set(manifest[p]) for p in ("train", "validation", "calibration"))) if manifest else set()
    manifest_missing = sorted(in_manifest - set(images))
    check("image count = 1000", len(images) == 1000, f"found {len(images)} .tif/.tiff files")
    check("every image has a label", not no_label, f"{len(no_label)} without label, e.g. {no_label[:5]}")
    check("every label has an image", not no_image, f"{len(no_image)} without image, e.g. {no_image[:5]}")
    check("no unexpected files or sub-folders in the image folder", not unexpected and not subdirs,
          f"unexpected files {unexpected[:5]}, sub-folders {subdirs[:5]}")
    check("no filenames differing only by case/extension", not clashes, f"{clashes[:5]}")
    check("every manifest entry has an image file", not manifest_missing, f"{len(manifest_missing)} missing, e.g. {manifest_missing[:5]}")
    show("Image folder vs label file (filenames only)", pd.DataFrame({"count": {
        "image files (.tif/.tiff)": len(images), "label keys": len(labels), "images without label": len(no_label),
        "labels without image": len(no_image), "unexpected files": len(unexpected), "sub-folders": len(subdirs),
        "case/extension clashes": len(clashes), "manifest entries without image": len(manifest_missing)}}))
    return True, ""


guarded("image folder readable", filenames_check)

In [ ]:
def weights_check():
    weights_dir = Path("weights/pretrained/radimagenet/resnet50")
    rows = []
    for line in (weights_dir / "CHECKSUMS.sha256").read_text().splitlines():
        if not line.strip():
            continue
        digest, name = line.split()[0], line.split()[-1].lstrip("*")
        path = weights_dir / name
        ok = path.is_file() and sha256_file(path) == digest
        rows.append({"file": name, "present": path.is_file(), "sha256 matches": ok})
        check(f"weights {name}", ok, "file missing" if not path.is_file() else "SHA-256 differs from CHECKSUMS.sha256")
    show("Converted weights vs recorded SHA-256", pd.DataFrame(rows))
    check("at least one weights file is recorded in CHECKSUMS.sha256", bool(rows))
    return True, ""


guarded("weights checksum file readable", weights_check)
show("Historical checkpoints found (listed only, never opened)",
     pd.DataFrame({"historical checkpoint": [p.as_posix() for p in sorted(Path("experiments").glob("00*/**/checkpoint/*.pt"))]}))

In [ ]:
summary = pd.DataFrame({"passed": CHECKS})
show("Check summary", summary)
failed = [k for k, ok in CHECKS.items() if not ok]
if failed:
    show("FAILED checks and why", pd.DataFrame({"detail": {k: DETAILS.get(k, "") for k in failed}}))
    raise RuntimeError(f"AUDIT FAILED: {len(failed)} of {len(CHECKS)} checks failed: {failed}")
print(f"AUDIT PASSED: all {len(CHECKS)} checks passed (development data only; the official test set was not accessed).")